## Market Place and Seller performance analytics 
The objective of this notebook is to evaluate seller performance across the Olist marketplace. 
This notebook aims to answer: 
1. which sellers contribute the most revenue?
2. which sellers process the largest number of orders ? 
3. which sellers receive the highest customer ratings ?
4. how is the seller network distrubuited geographically ?
5. Does marketplace revenue depend on a small number of sellers ? 


In [2]:
import warnings 
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns 

In [3]:

project_root = Path.cwd().parent.parent
data_dir = project_root / "datasets" / "Olist"


In [4]:
orders = pd.read_csv(data_dir/"olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp"])
order_items=pd.read_csv(data_dir/"olist_order_items_dataset.csv")
reviews= pd.read_csv(
    data_dir/"olist_order_reviews_dataset.csv"
)
sellers= pd.read_csv(
    data_dir/"olist_sellers_dataset.csv"
)

In [7]:
seller_df = (
    order_items
    .merge(
        orders,
        on="order_id",
        how="left"
    )
    .merge(
        reviews,
        on="order_id",
        how="left"
    )
    .merge(
        sellers,
        on="seller_id",
        how="left"
    )
)
seller_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,2017-09-29 00:00:00,97ca439bc427b48bc1cd7177abe71365,5.0,NaN,"Perfeito, produto entregue antes do combinado.",2017-09-21 00:00:00,2017-09-22 10:57:03,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,2017-05-15 00:00:00,7b07bacd811c4117b742569b04ce3580,4.0,NaN,NaN,2017-05-13 00:00:00,2017-05-15 11:34:13,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,2018-02-05 00:00:00,0c5b33dea94867d1ac402749e5438e8b,5.0,NaN,Chegou antes do prazo previsto e o produto sur...,2018-01-23 00:00:00,2018-01-23 16:06:31,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,2018-08-20 00:00:00,f4028d019cb58564807486a6aaf33817,4.0,NaN,NaN,2018-08-15 00:00:00,2018-08-15 16:39:01,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,2017-03-17 00:00:00,940144190dcba6351888cafa43f3a3a5,5.0,NaN,Gostei pois veio no prazo determinado .,2017-03-02 00:00:00,2017-03-03 10:54:59,87900,loanda,PR


In [8]:
# seller revenue 
seller_revenue =(
    seller_df
    .groupby("seller_id")
    .agg(
        Revenue=("price","sum"),
        Orders=("order_id","nunique")
    )
    .sort_values(
        "Revenue",
        ascending=False
    )
    .reset_index()
)
seller_revenue.head(10)

,seller_id,Revenue,Orders
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132
1,53243585a1d6dc2643021fd1853d8905,222776.05,358
2,4a3ca9315b744ce9f8e9374361493884,202999.12,1806
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585
4,7c67e1448b00f6e969d365cea6b010ab,189417.67,982
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,336
6,da8622b14eb17ae2831f4ac5b9dab84a,162723.37,1314
7,7a67c85e85bb2ce8582c35f2203ad736,142325.49,1160
8,1025f0e2d44d7041d6cf58b6550e0bfa,140513.14,915
9,955fee9216a65b617aa5c0531780ce60,135241.70,1287


In [10]:
fig = px.bar(
    seller_revenue.head(10),
    x="Revenue",
    y="Orders",
    orientation="h",
    title="top 10 sellers by revenue",
    template="plotly_white"
)
fig.update_yaxes(showticklabels=False)
fig.show()

In [11]:
# seller order volume 
seller_orders=(
    seller_df
    .groupby("seller_id")
    .agg(
        Orders=("order_id","nunique")
    )
    .sort_values(
        "Orders",
        ascending=False
    )
    .reset_index()
)


In [15]:
fig = px.bar(
    seller_orders.head(10),
    x="Orders",
    y="seller_id",
    orientation="h",
    title="Top sellers by orders",
    template="plotly_white"
)
fig.show()

In [16]:
# seller average order value 
seller_aov=(
    seller_df
    .groupby("seller_id")
    .agg(
        AverageOrderValue=("price","mean")
    )
    .sort_values(
        "AverageOrderValue",
        ascending=False
    )
    .reset_index()
)

In [17]:
fig = px.bar(
    seller_aov.head(10),
    x="AverageOrderValue",
    y="seller_id",
    orientation="h",
    title="Highest Average Order Value",
    template="plotly_white"
)
fig.update_yaxes(showticklabels=False)
fig.show()

In [22]:
# seller ratings 
seller_ratings=(
    seller_df
    .groupby("seller_id")
    .agg(AverageRating=("review_score","mean"),
         Reviews=("review_score", "count")
    )
    .query("Reviews>=20")
    .sort_values("AverageRating",ascending=False)
    .reset_index()
)

In [23]:
fig=px.bar(
    seller_ratings,
    x="AverageRating",
    y="seller_id",
    orientation="h",
    title="Highest Rated Sellers",
    template="plotly_white"
)
fig.update_yaxes(showticklabels=False)
fig.show()

In [28]:
# seller distribution 
seller_states=(
    sellers["seller_state"]
    .value_counts()
    .reset_index()
)
seller_states.columns=["state","sellers"]


In [31]:
fig=px.bar(
    seller_states,
    x="state",
    y="sellers",
    title="Seller Distribution by State",
    template="plotly_white"
)
fig.show()

In [35]:
# pareto-analysis : 80% of result is by 20% of the cause
# Sort sellers by revenue
pareto = seller_revenue.sort_values(
    "Revenue",
    ascending=False
).reset_index(drop=True)

# Revenue contribution
pareto["Revenue Share (%)"] = (
    pareto["Revenue"] /
    pareto["Revenue"].sum()
) * 100

# Cumulative revenue
pareto["Cumulative Revenue (%)"] = (
    pareto["Revenue Share (%)"]
    .cumsum()
)

# Percentage of sellers
pareto["Seller Percentage (%)"] = (
    (pareto.index + 1) /
    len(pareto)
) * 100

pareto.head()

,seller_id,Revenue,Orders,Revenue Share (%),Cumulative Revenue (%),Seller Percentage (%)
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,1.680881,1.680881,0.032310
1,53243585a1d6dc2643021fd1853d8905,222776.05,358,1.631829,3.312710,0.064620
2,4a3ca9315b744ce9f8e9374361493884,202999.12,1806,1.486964,4.799674,0.096931
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,1.421353,6.221027,0.129241
4,7c67e1448b00f6e969d365cea6b010ab,189417.67,982,1.387480,7.608507,0.161551


In [36]:
import plotly.graph_objects as go

fig = go.Figure()

# Revenue bars
fig.add_trace(
    go.Bar(
        x=pareto.index + 1,
        y=pareto["Revenue"],
        name="Seller Revenue"
    )
)

# Cumulative revenue line
fig.add_trace(
    go.Scatter(
        x=pareto.index + 1,
        y=pareto["Cumulative Revenue (%)"],
        mode="lines",
        name="Cumulative Revenue",
        yaxis="y2"
    )
)

fig.update_layout(

    title="Pareto Analysis of Seller Revenue",

    xaxis_title="Sellers (Ranked by Revenue)",

    yaxis=dict(
        title="Revenue"
    ),

    yaxis2=dict(
        title="Cumulative Revenue (%)",
        overlaying="y",
        side="right",
        range=[0,100]
    ),

    template="plotly_white"

)

fig.add_hline(
    y=80,
    line_dash="dash",
    line_color="red"
)

fig.show()

**Approximately 14% of sellers generate 80% of marketplace revenue, confirming a strong Pareto effect.**

In [34]:
seller_score = (
    seller_df
    .groupby("seller_id")
    .agg(
        Revenue=("price","sum"),
        Orders=("order_id","nunique"),
        Rating=("review_score","mean")
    )
)

seller_score["Score"] = (
    0.5*(
        seller_score["Revenue"]/
        seller_score["Revenue"].max()
    )
    +
    0.3*(
        seller_score["Orders"]/
        seller_score["Orders"].max()
    )
    +
    0.2*(
        seller_score["Rating"]/
        5
    )
)

seller_score.sort_values(
    "Score",
    ascending=False
).head(10)

,Revenue,Orders,Rating,Score
seller_id,,,,
4a3ca9315b744ce9f8e9374361493884,202999.12,1806,3.803931,0.886707
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,4.122822,0.848084
da8622b14eb17ae2831f4ac5b9dab84a,162723.37,1314,4.071429,0.730038
6560211a19b47992c3666cc44a7e94c0,123585.82,1854,3.909406,0.725659
53243585a1d6dc2643021fd1853d8905,222776.05,358,4.075980,0.706377
7c67e1448b00f6e969d365cea6b010ab,189417.67,982,3.348208,0.705552
fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,4.340206,0.691068
cc419e0650a3c5ba77189a1882b7556a,106555.98,1706,4.069575,0.671011
7a67c85e85bb2ce8582c35f2203ad736,142325.49,1160,4.234991,0.667216


# Marketplace & Seller Performance Insights

## 1. Revenue Concentration

- Marketplace revenue is highly concentrated among a relatively small number of sellers.
- The highest-performing seller generated approximately **R$229,473**, while several other sellers also generated revenues exceeding **R$180,000**.
- This indicates that a small group of sellers contributes disproportionately to the marketplace's total revenue.

---

## 2. Order Volume

- Seller order volumes vary significantly across the marketplace.
- The most active sellers processed more than **1,800 unique orders**, demonstrating substantial differences in seller scale.
- High order volume does not always translate into the highest revenue, indicating differences in product pricing and average transaction value.

---

## 3. Customer Ratings

- Most top-performing sellers maintain customer ratings close to **4 out of 5**, reflecting generally positive customer experiences.
- However, some high-revenue sellers have comparatively lower ratings, suggesting that commercial success does not always coincide with customer satisfaction.
- Monitoring customer ratings alongside sales performance is essential for maintaining marketplace quality.

---

## 4. Seller Geographic Distribution

- Sellers are concentrated in a limited number of Brazilian states, particularly São Paulo.
- This geographic concentration suggests operational dependence on a few major commercial regions.
- Expanding seller acquisition into underrepresented states could improve marketplace resilience and geographic coverage.

---

## 5. Seller Health Score

- The Seller Health Score successfully combines **Revenue (50%)**, **Order Volume (30%)**, and **Average Customer Rating (20%)** into a single performance indicator.
- Unlike revenue alone, the composite score identifies sellers who consistently perform well across multiple business dimensions.
- For example, seller **4a3ca9315b744ce9f8e9374361493884** ranks first overall despite not having the highest revenue, because of its exceptional order volume and strong customer ratings.
- This demonstrates that sustainable marketplace performance depends on balancing sales, operational efficiency, and customer satisfaction rather than maximizing revenue alone.

---

# Strategic Business Recommendations

### Prioritize Elite Sellers

Develop premium partnership programs, exclusive promotions, and operational support for sellers with the highest Seller Health Scores to maximize long-term marketplace growth.

---

### Support High-Volume Sellers

Sellers processing large numbers of orders should receive logistics and inventory support to maintain service quality during periods of high demand.

---

### Improve Customer Experience

High-revenue sellers with comparatively lower customer ratings should be targeted for quality improvement initiatives, including seller training, fulfillment optimization, and customer service enhancements.

---

### Expand the Seller Network

Recruit additional sellers from regions with low marketplace representation to reduce geographic concentration and improve nationwide coverage.

---

### Monitor Marketplace Health

Rather than evaluating sellers solely by revenue, use the Seller Health Score as a continuous KPI for marketplace monitoring. This provides a more balanced assessment by incorporating financial performance, operational activity, and customer satisfaction into a single decision-making metric.